# pyaegean neural-lemmatizer spike — GreBERTa + edit-tree head

Goal: beat stanza/CLTK on the hardest column, **unseen-form lemma** (pure-Python baseline 40.3%, stanza 62.8%) — by classifying the **same edit-tree label set** the pure-Python lemmatizer uses, but from a fine-tuned Ancient-Greek transformer.

**Loop:** set runtime to a **GPU** (an H100 is ideal; a T4 also works) → Run all → download `spike_model.zip` at the end → send it back for the torch-free local eval.

Precision auto-detects: **bf16 + TF32 + fused AdamW + batch 64 on H100/A100**, fp16 + batch 16 on a T4. Encoder: `bowphs/GreBerta` (Apache-2.0, Ancient-Greek RoBERTa). torch/transformers are used **only here**; production inference is onnxruntime-only.

In [ ]:
!nvidia-smi -L  # confirm the GPU (H100 ideal)
%pip -q install 'transformers>=4.46' 'datasets>=2.19' 'optimum[onnxruntime]>=1.20' 'accelerate>=0.30' onnx onnxruntime

## 0 · Detect GPU + precision

In [ ]:
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()  # Ampere/Hopper+
BS = 64 if USE_BF16 else 16
print(f'torch {torch.__version__} | CUDA {torch.version.cuda} | GPU {gpu}')
print(f'bf16={USE_BF16}  ->  batch_size={BS}, '
      f'precision={"bf16+tf32" if USE_BF16 else "fp16"}')

## 1 · Upload the data bundle
Upload **`spike_data.zip`** (zip of `experiments/neural_lemma_spike/data/`: `train.jsonl`, `dev.jsonl`, `labels.json`).

In [ ]:
import json, zipfile, pathlib
from google.colab import files
up = files.upload()  # pick spike_data.zip
zname = next(n for n in up if n.endswith('.zip'))
zipfile.ZipFile(zname).extractall('.')
DATA = pathlib.Path('data') if pathlib.Path('data/labels.json').exists() else pathlib.Path('.')
labels = json.loads((DATA / 'labels.json').read_text(encoding='utf-8'))['trees']
id2label = {i: k for i, k in enumerate(labels)}
label2id = {k: i for i, k in enumerate(labels)}
print('edit-tree labels:', len(labels))

## 2 · Load the training data

In [ ]:
from datasets import Dataset
def read_jsonl(p):
    return [json.loads(l) for l in open(p, encoding='utf-8')]
train_rows = read_jsonl(DATA / 'train.jsonl')
ds = Dataset.from_list([{'tokens': r['tokens'], 'tags': r['labels']} for r in train_rows])
print(ds)

## 3 · Tokenizer + model (`bowphs/GreBerta`)

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
MODEL = 'bowphs/GreBerta'
tokenizer = AutoTokenizer.from_pretrained(MODEL, add_prefix_space=True)
model = AutoModelForTokenClassification.from_pretrained(
    MODEL, num_labels=len(labels), id2label=id2label, label2id=label2id)

## 4 · Tokenize + align labels to the first sub-token of each word
Label only the first sub-token of each word, `-100` (ignored by the loss) elsewhere — matching the local eval's first-subword pooling.

In [ ]:
def align(batch):
    enc = tokenizer(batch['tokens'], is_split_into_words=True, truncation=True,
                    max_length=256)
    out = []
    for i, tags in enumerate(batch['tags']):
        word_ids = enc.word_ids(i)
        prev, row = None, []
        for wid in word_ids:
            if wid is None:
                row.append(-100)
            elif wid != prev:
                row.append(tags[wid])      # already -100 for pruned tokens
            else:
                row.append(-100)
            prev = wid
        out.append(row)
    enc['labels'] = out
    return enc
tok_ds = ds.map(align, batched=True, remove_columns=ds.column_names)
tok_ds = tok_ds.train_test_split(test_size=0.02, seed=0)  # small dev for best-epoch selection
print(tok_ds)

## 5 · Fine-tune (H100-tuned; keeps the best epoch by dev token-accuracy)
On an H100 this is a few minutes for 4 epochs; on a T4, longer. `load_best_model_at_end` uses the dev split so extra epochs can't overfit the kept checkpoint.

In [ ]:
import numpy as np
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification
collator = DataCollatorForTokenClassification(tokenizer)
def metrics(p):
    preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    preds = np.argmax(preds, axis=-1)
    gold = p.label_ids
    m = gold != -100
    return {'tok_acc': float((preds[m] == gold[m]).mean())}
args = TrainingArguments(
    output_dir='out', learning_rate=3e-5,
    per_device_train_batch_size=BS, per_device_eval_batch_size=BS * 2,
    num_train_epochs=4, weight_decay=0.01, warmup_ratio=0.06,
    bf16=USE_BF16, fp16=not USE_BF16,    # bf16 on H100/A100, fp16 on T4
    tf32=USE_BF16,                        # TF32 matmuls on Ampere+ (no-op on T4)
    optim='adamw_torch_fused',            # fused optimizer — faster on CUDA
    dataloader_num_workers=2,
    eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model='tok_acc', greater_is_better=True,
    logging_steps=50, report_to='none')
trainer = Trainer(model=model, args=args, train_dataset=tok_ds['train'],
                  eval_dataset=tok_ds['test'], data_collator=collator,
                  processing_class=tokenizer, compute_metrics=metrics)  # processing_class, not tokenizer=
trainer.train()
trainer.save_model('out_model'); tokenizer.save_pretrained('out_model')
print('in-notebook token-accuracy is a sanity check only; the real lemma number is the\n'
      'local torch-free eval_spike.py on dev.jsonl')

## 6 · Export to ONNX + int8 quantize (current optimum / onnxruntime APIs)
`optimum-cli export onnx` then `onnxruntime.quantization.quantize_dynamic`. If quantization errors on this version, the fp32 export ships instead — `eval_spike.py` runs either.

In [ ]:
import os
!optimum-cli export onnx --model out_model --task token-classification onnx_fp32
onnx_path = 'onnx_fp32/model.onnx'
try:
    from onnxruntime.quantization import quantize_dynamic, QuantType
    quantize_dynamic('onnx_fp32/model.onnx', 'onnx_fp32/model_int8.onnx', weight_type=QuantType.QInt8)
    onnx_path = 'onnx_fp32/model_int8.onnx'
    print('int8 quantized ->', onnx_path)
except Exception as e:
    print('quantization skipped (' + str(e)[:160] + '); shipping fp32')
for f in sorted(os.listdir('onnx_fp32')):
    print(f, os.path.getsize(os.path.join('onnx_fp32', f)) // 1024, 'KB')

## 7 · Package + download `spike_model.zip`

In [ ]:
import shutil
pathlib.Path('ship').mkdir(exist_ok=True)
shutil.copy(onnx_path, 'ship/model.onnx')
shutil.copy('onnx_fp32/tokenizer.json', 'ship/tokenizer.json')
shutil.copy(str(DATA / 'labels.json'), 'ship/labels.json')
shutil.make_archive('spike_model', 'zip', 'ship')
print('model.onnx', os.path.getsize('ship/model.onnx') // (1024 * 1024), 'MB')
files.download('spike_model.zip')

## Next
Send back `spike_model.zip`. Local, torch-free:
```
pip install onnxruntime tokenizers numpy
python eval_spike.py --model model.onnx --tokenizer tokenizer.json \
                     --labels data/labels.json --dev data/dev.jsonl
```
Success = **unseen lemma > 62.8%** on our own split → green-light the full multi-task backend.